# Qwen3-8B Coding Fine-tune — OpenCodeInstruct + QLoRA + Kaggle T4x2

This notebook trains a coding-specialized Qwen3-8B using QLoRA/SFT on a filtered slice of
**OpenCodeInstruct** (NVIDIA, 5M examples, CC BY 4.0), evaluates it against the untouched
base model on **HumanEval+** and **MBPP+**, and publishes the result to Hugging Face.

## Before you run anything — Kaggle settings
- **Settings (right panel) → Accelerator → GPU T4 x2**
- **Settings → Internet → On**
- **Add-ons → Secrets → add a secret named `HF_TOKEN`** with a Hugging Face **write**-access
  token (create one at hf.co/settings/tokens). Attach it to this notebook.
- Kaggle requires a **phone-verified account** before the GPU/Internet options even appear —
  do that first if you haven't.
- Quota reminder: **30 GPU-hours/week, reset weekly, 12-hour hard session cap.** Training will
  span multiple sessions — that's expected and handled below (checkpoints resume from Hugging Face).

## What this notebook does, in order
1. Loads Qwen3-8B-**Instruct** (4-bit) and benchmarks it, untouched, on HumanEval+/MBPP+ — your "before" numbers.
2. Streams and filters OpenCodeInstruct (no full 5M-row download) into a training set.
3. Writes a real training script and launches it with `torchrun` across both T4s (DDP) — real
   training from the first minute, no throwaway test run. The first checkpoint's speed tells you
   your real tokens/sec, which is what you use to plan the remaining sessions.
4. Re-runs the same benchmarks on your fine-tuned model for a before/after comparison.
5. Publishes the LoRA adapter (continuously, every checkpoint) and, at the end, a merged fp16
   model, to Hugging Face.

## Design decisions locked in for this version (why some things look simpler than earlier drafts)
- **OpenCodeInstruct only** — it's already a complete, test-verified, 5M-example coding SFT set.
  rStar-Coder is deliberately left out of v1; add it later as a mixed-in v2 experiment with
  `datasets.interleave_datasets` if you want the ablation story — nothing here needs to change to add it.
- **Streaming** avoids materializing the full dataset on Kaggle's disk before training starts. To be
  precise: it still transfers data progressively as you iterate — it does not mean zero network
  traffic, just no big upfront download-and-store step.
- **Deduplication** is exact-hash only, and only against examples that already *passed* the quality
  filter — so the in-memory hash set is bounded by your final retained dataset size (tens/hundreds
  of thousands of rows), not the raw 5M, which keeps it genuinely cheap.
- **Decontamination**: OpenCodeInstruct is a distinct, separately-sourced dataset from
  HumanEval/MBPP, so the eval sets are structurally separate already. This notebook documents that
  separation rather than running a live per-row streaming check against the benchmarks; treat that
  as an optional add-on, not a blocker.
- **Multi-GPU DDP via `torchrun`** is a real, currently-documented Unsloth workflow (their own DDP
  guide uses Qwen3-8B as the example command), not a workaround — this notebook uses that exact,
  standard pattern.
- **Checkpoints push to the Hugging Face Hub automatically** on every save, using the Trainer's
  built-in Hub integration — this saves full training state (model, optimizer, scheduler), not just
  weights, which is what makes a clean resume across Kaggle sessions possible.
- **Merging to fp16** uses Unsloth's documented `save_pretrained_merged(..., save_method="merged_16bit")`
  helper (this is the standard, supported path — it dequantizes properly rather than trying to merge
  directly into the 4-bit training weights). It's run as its own separate step after training, in a
  fresh session, so it has full RAM/VRAM headroom rather than competing with an active training run.


## Step 0 — Sanity check the environment

In [ ]:
# Confirms both T4s are visible before we install anything or spend any quota.
!nvidia-smi --query-gpu=index,name,memory.total,memory.used --format=csv


## Step 1 — Install dependencies

In [ ]:
%%capture
!pip install -U pip -q
!pip install "unsloth" -q
!pip install -U trl peft accelerate bitsandbytes datasets huggingface_hub -q
!pip install evalplus -q
# If unsloth's plain pip install ever lags a new model release, the fallback is:
#   pip install --upgrade --force-reinstall --no-cache-dir "git+https://github.com/unslothai/unsloth.git"


## Step 2 — Secrets, config, imports

In [ ]:
import os, random

from kaggle_secrets import UserSecretsClient
from huggingface_hub import login, HfApi

HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
login(HF_TOKEN)
os.environ["HF_TOKEN"] = HF_TOKEN

# ---- EDIT THESE TWO LINES ----
HF_USERNAME = "Maliktg7"
PROJECT_NAME = "qwen3-8b-opencodeinstruct"
# -------------------------------

HF_CHECKPOINT_REPO = f"{HF_USERNAME}/{PROJECT_NAME}-checkpoints"   # LoRA adapter, pushed every save
HF_MERGED_REPO      = f"{HF_USERNAME}/{PROJECT_NAME}-merged"        # fp16 merged model, pushed at the end

BASE_MODEL = "unsloth/Qwen3-8B-unsloth-bnb-4bit"   # Qwen3-8B-Instruct, pre-quantized 4-bit by Unsloth
MAX_SEQ_LENGTH = 2048
MIN_TEST_SCORE = 0.8   # OpenCodeInstruct's own average_test_score quality bar (0-1 scale)

random.seed(3407)

api = HfApi()
api.create_repo(HF_CHECKPOINT_REPO, exist_ok=True, private=True)
api.create_repo(HF_MERGED_REPO, exist_ok=True, private=True)
print("Checkpoints will push to:", HF_CHECKPOINT_REPO)
print("Merged model will push to:", HF_MERGED_REPO)


## Phase 1 — Baseline: benchmark the untouched model first

This is inference, not training — cheap, and it's the only source of "before" numbers you'll
have later. It runs regardless of everything else, so do it now while the harness is easy to debug.


In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = BASE_MODEL,
    max_seq_length = MAX_SEQ_LENGTH,
    dtype = None,          # auto-detects float16 on T4, bfloat16 on newer GPUs
    load_in_4bit = True,
)
FastLanguageModel.for_inference(model)
print("Loaded", BASE_MODEL)


In [ ]:
from evalplus.data import get_human_eval_plus, get_mbpp_plus, write_jsonl

# IMPORTANT: Qwen3 defaults to "thinking mode" (enable_thinking=True), and Qwen's own docs warn
# against combining thinking mode with greedy decoding ("can lead to performance degradation and
# endless repetitions"). We want plain, direct code completions (matching the training data, which
# has no reasoning traces) and reproducible greedy pass@1, so thinking mode is explicitly OFF here.
def generate_solution(prompt, max_new_tokens=512):
    instruction = (
        "Complete the following Python function. "
        "Respond with a single Python code block containing the complete function "
        "(including the signature) and nothing else.\n\n```python\n" + prompt + "\n```"
    )
    convo = [{"role": "user", "content": instruction}]
    inputs = tokenizer.apply_chat_template(
        convo, tokenize=True, add_generation_prompt=True, enable_thinking=False,
        return_tensors="pt",
    ).to(model.device)
    out = model.generate(
        input_ids=inputs, max_new_tokens=max_new_tokens,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )
    text = tokenizer.decode(out[0][inputs.shape[-1]:], skip_special_tokens=True)
    return text

def run_generation(problems, out_path):
    samples = []
    for task_id, problem in problems.items():
        solution = generate_solution(problem["prompt"])
        samples.append({"task_id": task_id, "solution": solution})
    write_jsonl(out_path, samples)
    print(f"Wrote {len(samples)} samples to {out_path}")

os.makedirs("eval_baseline", exist_ok=True)
run_generation(get_human_eval_plus(), "eval_baseline/humaneval_samples.jsonl")
run_generation(get_mbpp_plus(),       "eval_baseline/mbpp_samples.jsonl")


In [ ]:
import glob

# evalplus.sanitize strips markdown fences / stray prose from chat-model output so the
# remaining code is actually executable; evalplus.evaluate then scores it.
!evalplus.sanitize --samples eval_baseline/humaneval_samples.jsonl
!evalplus.sanitize --samples eval_baseline/mbpp_samples.jsonl

def find_sanitized(path):
    stem = path.rsplit(".jsonl", 1)[0]
    matches = glob.glob(stem + "*sanitiz*")
    return matches[0] if matches else path

he_file = find_sanitized("eval_baseline/humaneval_samples.jsonl")
mbpp_file = find_sanitized("eval_baseline/mbpp_samples.jsonl")

print("Evaluating HumanEval+ ...")
!evalplus.evaluate --dataset humaneval --samples {he_file}
print("Evaluating MBPP+ ...")
!evalplus.evaluate --dataset mbpp --samples {mbpp_file}

**Write down the pass@1 numbers printed above (or screenshot them) — these are your baseline.**
You'll compare the fine-tuned model against these exact numbers in Phase 5. If you want to free
GPU memory before moving on, restart the session now; nothing above needs to be kept in memory.


## Phase 2 — Build the streaming training set (preview only, no GPU needed)

Streams OpenCodeInstruct, keeps only high-scoring examples (`average_test_score >= MIN_TEST_SCORE`),
drops exact duplicates, formats each into Qwen3's chat template, and drops anything that doesn't
fit in `MAX_SEQ_LENGTH`. This cell just previews a few formatted rows so you can eyeball the
pipeline before committing a training session to it — this preview runs on CPU and costs no GPU quota.


In [ ]:
from datasets import load_dataset

def build_training_stream(tokenizer, min_test_score=MIN_TEST_SCORE, max_seq_length=MAX_SEQ_LENGTH):
    raw = load_dataset("nvidia/OpenCodeInstruct", split="train", streaming=True)

    def passes_quality(ex):
        try:
            return float(ex["average_test_score"]) >= min_test_score
        except (TypeError, ValueError):
            return False

    quality = raw.filter(passes_quality)

    seen_hashes = set()   # bounded by the RETAINED set size, not the raw 5M rows
    def is_new(ex):
        h = hash(ex["input"])
        if h in seen_hashes:
            return False
        seen_hashes.add(h)
        return True

    deduped = quality.filter(is_new)

    def format_example(ex):
        convo = [
            {"role": "user", "content": ex["input"]},
            {"role": "assistant", "content": ex["output"]},
        ]
        text = tokenizer.apply_chat_template(convo, tokenize=False, add_generation_prompt=False)
        return {"text": text}

    formatted = deduped.map(format_example)

    def fits_length(ex):
        return len(tokenizer(ex["text"])["input_ids"]) <= max_seq_length

    return formatted.filter(fits_length)

preview_stream = build_training_stream(tokenizer)
for i, row in enumerate(preview_stream.take(3)):
    print(f"--- example {i} ({len(tokenizer(row['text'])['input_ids'])} tokens) ---")
    print(row["text"][:600], "...\n")


## Phase 3 — Write the training script

This has to be a standalone `.py` file (not notebook cells) because multi-GPU DDP is launched
with `torchrun`, which spawns separate processes that each import and run this file.


In [ ]:
%%writefile train.py
"""
Standalone QLoRA/SFT training script for Qwen3-8B on OpenCodeInstruct.
Launch with:  torchrun --nproc_per_node=2 train.py [--resume] [--max_steps N]
"""
import os
import argparse

from datasets import load_dataset, Dataset
from unsloth import FastLanguageModel, is_bfloat16_supported
from trl import SFTConfig, SFTTrainer
from huggingface_hub import snapshot_download

parser = argparse.ArgumentParser()
parser.add_argument("--resume", action="store_true", help="Pull the latest checkpoint from the Hub before training")
parser.add_argument("--max_steps", type=int, default=500, help="Raise this once you know your real tokens/sec")
parser.add_argument("--min_test_score", type=float, default=0.8)
parser.add_argument("--per_device_batch_size", type=int, default=2)
parser.add_argument("--grad_accum", type=int, default=8)
args = parser.parse_args()

BASE_MODEL = "unsloth/Qwen3-8B-unsloth-bnb-4bit"
MAX_SEQ_LENGTH = 2048
HF_CHECKPOINT_REPO = os.environ["HF_CHECKPOINT_REPO"]
HF_TOKEN = os.environ.get("HF_TOKEN")

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = BASE_MODEL,
    max_seq_length = MAX_SEQ_LENGTH,
    dtype = None,
    load_in_4bit = True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                       "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 32,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

def build_training_stream(tokenizer, min_test_score, max_seq_length, max_examples=20000):
    raw = load_dataset("nvidia/OpenCodeInstruct", split="train", streaming=True)
    seen_hashes = set()
    rows = []
    for ex in raw:
        if len(rows) >= max_examples:
            break
        try:
            score = float(ex.get("average_test_score", 0))
        except (TypeError, ValueError):
            continue
        if score < min_test_score:
            continue

        inp = ex.get("input", "")
        h = hash(inp)
        if h in seen_hashes:
            continue
        seen_hashes.add(h)

        convo = [
            {"role": "user", "content": inp},
            {"role": "assistant", "content": ex.get("output", "")},
        ]
        text = tokenizer.apply_chat_template(convo, tokenize=False, add_generation_prompt=False)

        if len(tokenizer(text)["input_ids"]) <= max_seq_length:
            rows.append({"text": text})
    return Dataset.from_list(rows)

train_stream = build_training_stream(tokenizer, args.min_test_score, MAX_SEQ_LENGTH)

if args.resume:
    try:
        snapshot_download(repo_id=HF_CHECKPOINT_REPO, local_dir="outputs", token=HF_TOKEN)
        print("Pulled latest checkpoint from", HF_CHECKPOINT_REPO)
    except Exception as e:
        print("No checkpoint found on the Hub yet, starting fresh:", e)
        args.resume = False

sft_config = SFTConfig(
    output_dir = "outputs",
    per_device_train_batch_size = args.per_device_batch_size,
    gradient_accumulation_steps = args.grad_accum,
    warmup_steps = 20,
    max_steps = args.max_steps,
    learning_rate = 2e-4,
    fp16 = not is_bfloat16_supported(),
    bf16 = is_bfloat16_supported(),
    logging_steps = 10,
    optim = "adamw_8bit",
    weight_decay = 0.01,
    lr_scheduler_type = "cosine",
    seed = 3407,
    save_strategy = "steps",
    save_steps = 200,
    save_total_limit = 3,
    dataset_text_field = "text",
    max_seq_length = MAX_SEQ_LENGTH,
    ddp_find_unused_parameters = False,
    push_to_hub = True,
    hub_model_id = HF_CHECKPOINT_REPO,
    hub_strategy = "every_save",
    hub_token = HF_TOKEN,
    report_to = "none",
)

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_stream,
    args = sft_config,
)

trainer_stats = trainer.train(resume_from_checkpoint=args.resume)
print(trainer_stats)

## Phase 4 — Launch real training (this IS session 1, not a test)

Run this cell as-is for your **first** session. Watch the first few minutes: confirm both GPUs
show load in `nvidia-smi` (open a terminal tab, or just trust the throughput number below), and
that the loss printed every 10 steps looks like a real number, not `nan`. Everything from step 1
onward is genuine training — nothing here gets thrown away.

`--max_steps 500` below is a deliberately modest starting point. Once this session ends, look at
how many steps you actually completed vs. wall-clock time, and raise `--max_steps` for the next
session accordingly so you use your full 12-hour window.

**Starting session 2 (and every session after)**: a new Kaggle session is a brand-new, empty
container — `train.py` and all installs are gone. Re-run, in order: Step 0 (sanity check) → Step 1
(installs) → Step 2 (secrets/config) → Phase 3 (`%%writefile train.py`) → then the launch cell
below **with `--resume` added** so it pulls the latest checkpoint before continuing. Skipping
`--resume` after session 1 silently starts over from scratch.


In [ ]:
os.environ["HF_CHECKPOINT_REPO"] = HF_CHECKPOINT_REPO

# Session 1:
!torchrun --nproc_per_node=2 train.py --max_steps 500

# Session 2 onward, uncomment and use instead (adjust max_steps as you learn your real throughput):
# !torchrun --nproc_per_node=2 train.py --resume --max_steps 3000


## Phase 5 — Final evaluation: same benchmarks, baseline vs. fine-tuned

Run this once training is done (or whenever you want to check progress against the baseline).
It reloads the latest checkpoint from the Hub, so it works in a fresh session too — **but if this
is a new Kaggle session, re-run Steps 0–2 above first** (installs + secrets + `HF_CHECKPOINT_REPO`
etc. need to exist in this kernel).


In [ ]:
# Self-contained: re-imports everything this phase needs, since it may run in a fresh session.
import os, glob
from evalplus.data import get_human_eval_plus, get_mbpp_plus, write_jsonl
from unsloth import FastLanguageModel

def find_sanitized(path):
    stem = path.rsplit(".jsonl", 1)[0]
    matches = glob.glob(stem + "*sanitiz*")
    return matches[0] if matches else path

ft_model, ft_tokenizer = FastLanguageModel.from_pretrained(
    model_name = HF_CHECKPOINT_REPO,
    max_seq_length = MAX_SEQ_LENGTH,
    dtype = None,
    load_in_4bit = True,
)
FastLanguageModel.for_inference(ft_model)
print("Loaded fine-tuned checkpoint from", HF_CHECKPOINT_REPO)


In [ ]:
# Same enable_thinking=False / greedy-decoding choice as Phase 1, for a fair apples-to-apples comparison.
def generate_solution_ft(prompt, max_new_tokens=512):
    instruction = (
        "Complete the following Python function. "
        "Respond with a single Python code block containing the complete function "
        "(including the signature) and nothing else.\n\n```python\n" + prompt + "\n```"
    )
    convo = [{"role": "user", "content": instruction}]
    inputs = ft_tokenizer.apply_chat_template(
        convo, tokenize=True, add_generation_prompt=True, enable_thinking=False,
        return_tensors="pt",
    ).to(ft_model.device)
    out = ft_model.generate(
        input_ids=inputs, max_new_tokens=max_new_tokens,
        do_sample=False,
        pad_token_id=ft_tokenizer.eos_token_id,
    )
    return ft_tokenizer.decode(out[0][inputs.shape[-1]:], skip_special_tokens=True)

def run_generation_ft(problems, out_path):
    samples = []
    for task_id, problem in problems.items():
        samples.append({"task_id": task_id, "solution": generate_solution_ft(problem["prompt"])})
    write_jsonl(out_path, samples)
    print(f"Wrote {len(samples)} samples to {out_path}")

os.makedirs("eval_finetuned", exist_ok=True)
run_generation_ft(get_human_eval_plus(), "eval_finetuned/humaneval_samples.jsonl")
run_generation_ft(get_mbpp_plus(),       "eval_finetuned/mbpp_samples.jsonl")


In [ ]:
!evalplus.sanitize --samples eval_finetuned/humaneval_samples.jsonl
!evalplus.sanitize --samples eval_finetuned/mbpp_samples.jsonl

he_file_ft = find_sanitized("eval_finetuned/humaneval_samples.jsonl")
mbpp_file_ft = find_sanitized("eval_finetuned/mbpp_samples.jsonl")

print("=== FINE-TUNED: HumanEval+ ===")
!evalplus.evaluate --dataset humaneval --samples {he_file_ft}
print("=== FINE-TUNED: MBPP+ ===")
!evalplus.evaluate --dataset mbpp --samples {mbpp_file_ft}
print()
print("Compare these pass@1 numbers against the baseline you recorded in Phase 1.")


## Phase 6 — Publish: merge to fp16 and push everything to Hugging Face

Run this as its own step after training is fully done, ideally in a fresh session/kernel restart
so the merge has full RAM headroom instead of competing with an active training run. The LoRA
adapter itself is already on the Hub continuously from Phase 4 (`HF_CHECKPOINT_REPO`) — this step
adds the merged, adapter-free fp16 model as a second, more convenient artifact.

**New Kaggle session? Re-run Steps 0–2 above first** (installs + secrets + `HF_TOKEN`/`HF_CHECKPOINT_REPO`/
`HF_MERGED_REPO`/`api` need to exist in this kernel before the cells below will work).


In [ ]:
from unsloth import FastLanguageModel

merge_model, merge_tokenizer = FastLanguageModel.from_pretrained(
    model_name = HF_CHECKPOINT_REPO,
    max_seq_length = MAX_SEQ_LENGTH,
    dtype = None,
    load_in_4bit = True,
)

# Standard, documented Unsloth path: dequantizes properly rather than merging into 4-bit weights directly.
merge_model.save_pretrained_merged("merged_model", merge_tokenizer, save_method = "merged_16bit")
merge_model.push_to_hub_merged(HF_MERGED_REPO, merge_tokenizer, save_method = "merged_16bit", token = HF_TOKEN)
print("Merged fp16 model pushed to", HF_MERGED_REPO)

# Make both repos public once you're happy with the results (they were created private above):
# api.update_repo_visibility(HF_CHECKPOINT_REPO, private=False)
# api.update_repo_visibility(HF_MERGED_REPO, private=False)


In [ ]:
# Minimal model card — fill in the bracketed numbers from Phase 1 / Phase 5, then push.
model_card = f"""---
license: apache-2.0
base_model: unsloth/Qwen3-8B
tags:
  - qlora
  - sft
  - code
datasets:
  - nvidia/OpenCodeInstruct
---

# {PROJECT_NAME}

Qwen3-8B fine-tuned with QLoRA/SFT on a filtered, quality-scored slice of
[OpenCodeInstruct](https://huggingface.co/datasets/nvidia/OpenCodeInstruct) (CC BY 4.0),
trained on 2x NVIDIA T4 GPUs (Kaggle) with PyTorch DDP via `torchrun`.

## Results (pass@1)

| Benchmark | Base Qwen3-8B | Fine-tuned |
|---|---|---|
| HumanEval+ | <FILL IN from Phase 1> | <FILL IN from Phase 5> |
| MBPP+      | <FILL IN from Phase 1> | <FILL IN from Phase 5> |

## Training details
- Base model: unsloth/Qwen3-8B-unsloth-bnb-4bit
- Method: QLoRA (r=16, alpha=32) + SFT
- Data: OpenCodeInstruct, filtered to average_test_score >= {MIN_TEST_SCORE}, exact-deduplicated
- Hardware: 2x Tesla T4 (Kaggle), DDP via torchrun
- Sequence length: {MAX_SEQ_LENGTH}

## Limitations
Trained on synthetic, LLM-generated instruction data; inherits any biases or gaps present in
OpenCodeInstruct. Evaluated only on HumanEval+/MBPP+ — results may not generalize to other
coding benchmarks or real-world repositories.
"""

with open("README.md", "w") as f:
    f.write(model_card)

api.upload_file(
    path_or_fileobj = "README.md",
    path_in_repo = "README.md",
    repo_id = HF_MERGED_REPO,
    token = HF_TOKEN,
)
print("Model card pushed to", HF_MERGED_REPO)


## Done — what's on Hugging Face now

- `HF_CHECKPOINT_REPO` — the LoRA adapter, pushed continuously during training (also your
  cross-session resume point).
- `HF_MERGED_REPO` — the standalone fp16 model plus the model card.

Next, outside this notebook: push the data-prep/training code (this notebook, or the extracted
`train.py`) to a public GitHub repo, and cross-link it with the Hugging Face model card.


In [ ]:
!pip install -q unsloth gradio trl peft accelerate bitsandbytes

In [ ]:
import os
from huggingface_hub import HfApi
from kaggle_secrets import UserSecretsClient

# Fetch token
try:
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN")

api = HfApi(token=HF_TOKEN)

# Make the merged model public
api.update_repo_settings(
    repo_id="Maliktg7/qwen3-8b-opencodeinstruct-merged",
    private=False
)

# Optional: Make the checkpoints repo public as well
api.update_repo_settings(
    repo_id="Maliktg7/qwen3-8b-opencodeinstruct-checkpoints",
    private=False
)

print("Repositories are now public!")

In [ ]:
import os
import torch
import gradio as gr
from unsloth import FastLanguageModel

# 1. Retrieve HF Token from Kaggle Secrets or Environment Variables
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN")

MODEL_ID = "Maliktg7/qwen3-8b-opencodeinstruct-merged"
MAX_SEQ_LENGTH = 2048

# 2. Load Model & Tokenizer in Fast 4-bit Mode via Unsloth (~5.5 GB VRAM)
print("Loading model with Unsloth 4-bit fast loader...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_ID,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,          # Auto-selects float16 on T4 GPUs
    load_in_4bit=True,   # Saves RAM/VRAM
    token=HF_TOKEN
)

# 3. Enable Unsloth Fast Inference Acceleration Kernels
FastLanguageModel.for_inference(model)
print("Model loaded successfully with Unsloth fast inference!")

# 4. Define Generation Handler for Gradio Chat UI
def chat_predict(message, history):
    messages = []
    for user_msg, bot_msg in history:
        messages.append({"role": "user", "content": user_msg})
        if bot_msg:
            messages.append({"role": "assistant", "content": bot_msg})

    messages.append({"role": "user", "content": message})

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    outputs = model.generate(
        input_ids=inputs.input_ids,
        max_new_tokens=1024,
        temperature=0.2,
        do_sample=False,  # Greedy decoding for consistent code generation
        pad_token_id=tokenizer.eos_token_id
    )

    response = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    return response

# 5. Launch Gradio Interface
demo = gr.ChatInterface(
    fn=chat_predict,
    title="🤖 Qwen3-8B Fast Code Assistant (Unsloth 4-bit)",
    description="Interactive evaluation interface for `Maliktg7/qwen3-8b-opencodeinstruct-merged`.",
    examples=[
        "Write a Python function to reverse a string without built-in functions.",
        "Implement an LRU Cache class in Python with O(1) get and put operations.",
        "Create an async rate-limited task processor in Python using asyncio and Semaphore."
    ]
)

demo.launch(share=True, debug=True)

In [ ]:
import os

# 1. Force PyTorch/Unsloth to use a single GPU (cuda:0)
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import torch
import gradio as gr
from unsloth import FastLanguageModel

# 2. Retrieve HF Token from Kaggle Secrets or Environment
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN")

MODEL_ID = "Maliktg7/qwen3-8b-opencodeinstruct-merged"
MAX_SEQ_LENGTH = 2048

# 3. Load Model on single GPU (cuda:0)
print("Loading model on GPU 0...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_ID,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,          # Auto-detects float16 on T4 GPUs
    load_in_4bit=True,   # ~5.5 GB VRAM footprint
    token=HF_TOKEN,
    device_map="cuda:0"  # Explicitly force single GPU
)

FastLanguageModel.for_inference(model)

if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id

print("Model loaded successfully on single GPU!")

# 4. System Presets
SYSTEM_PRESETS = {
    "⚡ Expert Python Developer": "You are an expert Python developer. Provide clean, highly efficient, production-ready code with concise docstrings and type hints.",
    "🛡️ Safe & Robust Architecture": "You are a senior software architect focusing on thread-safety, edge-case validation, exception handling, and robust memory management.",
    "🎯 Minimalist Competitive Coder": "Provide the most optimal, space and time efficient O(N) solution without conversational text."
}

# 5. Generation Handler
def generate_response(message, history, system_preset, custom_system, temperature, top_p, max_tokens):
    sys_prompt = custom_system if custom_system.strip() else SYSTEM_PRESETS.get(system_preset, "")

    messages = []
    if sys_prompt:
        messages.append({"role": "system", "content": sys_prompt})

    for user_msg, bot_msg in history:
        messages.append({"role": "user", "content": user_msg})
        if bot_msg:
            messages.append({"role": "assistant", "content": bot_msg})

    messages.append({"role": "user", "content": message})

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    attention_mask = inputs.get("attention_mask", torch.ones_like(inputs.input_ids))

    do_sample = temperature > 0.0

    outputs = model.generate(
        input_ids=inputs.input_ids,
        attention_mask=attention_mask,
        max_new_tokens=int(max_tokens),
        temperature=temperature if do_sample else 1.0,
        top_p=top_p if do_sample else 1.0,
        do_sample=do_sample,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id
    )

    response = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    return response

# 6. Gradio Studio Interface
custom_css = """
.main-title {
    text-align: center;
    background: linear-gradient(135deg, #6366f1 0%, #a855f7 100%);
    -webkit-background-clip: text;
    -webkit-text-fill-color: transparent;
    font-size: 2.2em;
    font-weight: 800;
    margin-bottom: 0.2em;
}
.subtitle {
    text-align: center;
    color: #64748b;
    font-size: 1.05em;
    margin-bottom: 1.2em;
}
"""

theme = gr.themes.Soft(
    primary_hue="indigo",
    secondary_hue="purple",
    neutral_hue="slate",
    font=[gr.themes.GoogleFont("Inter"), "ui-sans-serif", "system-ui"]
)

with gr.Blocks(theme=theme, css=custom_css, title="Qwen3-8B Code Studio") as demo:
    gr.HTML("""
        <div class="main-title">🚀 Qwen3-8B Code Studio</div>
        <div class="subtitle">Fine-tuned on OpenCodeInstruct • Accelerated by Unsloth 4-bit Engine</div>
    """)

    with gr.Row():
        with gr.Column(scale=1, min_width=300):
            gr.Markdown("### ⚙️ Persona & System Control")

            system_preset = gr.Dropdown(
                choices=list(SYSTEM_PRESETS.keys()),
                value="⚡ Expert Python Developer",
                label="Developer Persona"
            )

            custom_system = gr.Textbox(
                label="Custom System Prompt (Optional)",
                placeholder="Override default instructions...",
                lines=2
            )

            with gr.Accordion("🎛️ Hyperparameters", open=True):
                temperature = gr.Slider(
                    minimum=0.0, maximum=1.0, value=0.2, step=0.05,
                    label="Temperature",
                    info="0.0 = Precise & Deterministic; >0.5 = Creative"
                )
                top_p = gr.Slider(
                    minimum=0.1, maximum=1.0, value=0.95, step=0.05,
                    label="Top-P (Nucleus Sampling)"
                )
                max_tokens = gr.Slider(
                    minimum=128, maximum=2048, value=1024, step=128,
                    label="Max Generation Length"
                )

        with gr.Column(scale=3):
            chatbot = gr.Chatbot(
                height=520,
                show_copy_button=True,
                bubble_full_width=False
            )

            with gr.Row():
                msg = gr.Textbox(
                    placeholder="Ask a coding question, request an algorithm, or paste code to debug...",
                    show_label=False,
                    scale=8,
                    container=False
                )
                submit_btn = gr.Button("Send 🚀", variant="primary", scale=1)

            with gr.Row():
                clear_btn = gr.ClearButton([msg, chatbot], value="🗑️ Clear Chat")

            gr.Examples(
                examples=[
                    "Write a Python function for an LRU Cache with O(1) operations.",
                    "Implement a sliding window maximum using collections.deque.",
                    "Write an async task processor with rate limiting using asyncio.Semaphore.",
                    "Design a thread-safe bounded queue using threading.Condition."
                ],
                inputs=msg,
                label="💡 Quick Example Prompts"
            )

    def user_turn(user_message, history):
        return "", history + [[user_message, None]]

    def bot_turn(history, system_preset, custom_system, temperature, top_p, max_tokens):
        user_message = history[-1][0]
        bot_response = generate_response(
            user_message, history[:-1], system_preset, custom_system, temperature, top_p, max_tokens
        )
        history[-1][1] = bot_response
        return history

    msg.submit(
        user_turn, [msg, chatbot], [msg, chatbot], queue=False
    ).then(
        bot_turn, [chatbot, system_preset, custom_system, temperature, top_p, max_tokens], chatbot
    )

    submit_btn.click(
        user_turn, [msg, chatbot], [msg, chatbot], queue=False
    ).then(
        bot_turn, [chatbot, system_preset, custom_system, temperature, top_p, max_tokens], chatbot
    )

demo.queue().launch(share=True, debug=True)

Loading model on GPU 0...
==((====))==  Unsloth 2026.9.11: Fast Qwen3 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

Model loaded successfully on single GPU!


/tmp/ipykernel_820/3960383276.py:111: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=theme, css=custom_css, title="Qwen3-8B Code Studio") as demo:
/tmp/ipykernel_820/3960383276.py:111: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(theme=theme, css=custom_css, title="Qwen3-8B Code Studio") as demo:
/tmp/ipykernel_820/3960383276.py:149: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot = gr.Chatbot(
/tmp/ipykernel_820/3960383276.py:149: DeprecationWarning: The 'show_copy_butt

* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://3fcf5d53567ee620bd.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Both `max_new_tokens` (=1024) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=1024) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=1024) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=1024) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_

In [ ]:
!git config --global user.name "mudassar2224"
!git config --global user.email "m.mudassar22222@gmail.com"

# 1. Clone your repo (replace <YOUR_GITHUB_TOKEN> with a Personal Access Token from GitHub)
!git clone https://<YOUR_GITHUB_TOKEN>@github.com/mudassar2224/Fine_tune_qwen3-8b-opencodeinstruct-finetune.git

# 2. Copy the current notebook to the cloned repo folder
!cp /kaggle/working/__notebook_source__.ipynb Fine_tune_qwen3-8b-opencodeinstruct-finetune/qwen3_8b_coding_fine_tune.ipynb

# 3. Commit and push
%cd Fine_tune_qwen3-8b-opencodeinstruct-finetune
!git add qwen3_8b_coding_fine_tune.ipynb
!git commit -m "Add Qwen3-8B fine-tuning notebook"
!git push origin main